# 01 — Exploration des fichiers Freddie Mac (millésimes 2007 et 2015)
Ce notebook part des fichiers texte bruts de la Single-Family Loan-Level Dataset (échantillons de 50 000 prêts par millésime) et :
1. les convertit en CSV avec les noms de colonnes officiels ;
2. contrôle la cohérence entre fichier d'origination (un prêt = une ligne) et fichier de performance (un prêt = une ligne par mois) ;
3. compte les défauts selon la règle de `docs/regle_defaut.md` et mesure l'effet de la forbearance de 2020 ;
4. identifie les défauts avec une perte enregistrée, utilisables pour la LGD ;
5. mesure la part de FICO non disponible.

**Aucune ligne de données n'est affichée** : la licence Freddie Mac n'autorise pas leur redistribution. Seuls des comptages et des taux apparaissent.

## 0. Imports et chemins

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import pandas as pd
from pathlib import Path

RACINE = Path.cwd().parent
sys.path.append(str(RACINE / "src"))

from validation_toolkit.credit.loaders.freddie_io import names_from_excel, txt_to_csv
from validation_toolkit.credit.data_quality import controle_comptages, part_fico_manquant
from validation_toolkit.credit.defaults import (dates_defaut, marquer_forbearance, resume_defauts,
                                                sortie_des_defauts, marquer_defaut_souscription)
from validation_toolkit.credit.lgd import echantillon_lgd

RAW = RACINE / "data" / "raw"
INTERIM = RACINE / "data" / "interim"

## 1. Conversion texte → CSV (à faire une seule fois)
Les fichiers texte n'ont pas d'en-tête (séparateur `|`). Les noms de colonnes viennent du fichier de description `file_layout_july_2026.xlsx` (feuille 0 : origination, 31 colonnes ; feuille 1 : performance, 35 colonnes). Mettre `CONVERTIR = True` pour régénérer les CSV.

In [2]:
CONVERTIR = False

if CONVERTIR:
    layout = RAW / "file_layout_july_2026.xlsx"
    cols_orig = names_from_excel(layout, sheet_name=0, column=1, skip_rows=2)
    cols_perf = names_from_excel(layout, sheet_name=1, column=1, skip_rows=2)
    for an in (2007, 2015):
        txt_to_csv(RAW / f"sample_{an}/sample_orig_{an}.txt", cols_orig, INTERIM / f"orig_{an}.csv")
        txt_to_csv(RAW / f"sample_{an}/sample_perf_{an}.txt", cols_perf, INTERIM / f"perf_{an}.csv")

## 2. Chargement des CSV
Tout est lu en texte (`dtype=str`) pour ne perdre aucun code spécial (9999, 999, RA, XX…). On n'affiche que les dimensions.

In [3]:
orig_2007 = pd.read_csv(INTERIM / "orig_2007.csv", dtype=str, keep_default_na=False)
perf_2007 = pd.read_csv(INTERIM / "perf_2007.csv", dtype=str, keep_default_na=False)

orig_2015 = pd.read_csv(INTERIM / "orig_2015.csv", dtype=str, keep_default_na=False)
perf_2015 = pd.read_csv(INTERIM / "perf_2015.csv", dtype=str, keep_default_na=False)

print(orig_2007.shape, perf_2007.shape)

(50000, 31) (3015985, 35)


## 3. Contrôles de cohérence origination / performance
Chaque prêt doit figurer une fois en origination et sur plusieurs lignes mensuelles en performance (environ 60 à 70 lignes en moyenne), sans prêt orphelin d'un côté ou de l'autre.

In [4]:
# millésime 2007
res = controle_comptages(INTERIM / "orig_2007.csv", INTERIM / "perf_2007.csv", id_col="Loan Identifier")

lignes_origination               50,000
prets_origination_uniques        50,000
lignes_performance               3,015,985
prets_performance_uniques        50,000
lignes_par_pret_moyenne          60.32
prets_orig_sans_perf             0
prets_perf_sans_orig             0


In [5]:
# millésime 2015
res = controle_comptages(INTERIM / "orig_2015.csv", INTERIM / "perf_2015.csv", id_col="Loan Identifier")

lignes_origination               50,000
prets_origination_uniques        50,000
lignes_performance               3,501,425
prets_performance_uniques        50,000
lignes_par_pret_moyenne          70.03
prets_orig_sans_perf             0
prets_perf_sans_orig             0


## 4. Défauts selon la règle v1
Premier mois où le statut d'impayés vaut 03 ou plus (90 jours et plus), ou RA (bien saisi), ou le code de sortie est dans {02, 03, 09, 15}. Détail dans `docs/regle_defaut.md`.

In [6]:
df_def_2007 = dates_defaut(INTERIM / "perf_2007.csv")
n_2007, par_annee_2007 = resume_defauts(df_def_2007, n_prets=50_000)

défauts : 8,265 sur 50,000 prêts, taux = 16.53%

par déclencheur du premier mois :
declencheur
statut    8242
sortie      21
RA           2
Name: count, dtype: int64

par année calendaire de défaut :
date_defaut
2007      82
2008     925
2009    2353
2010    1753
2011    1087
2012     734
2013     427
2014     226
2015     159
2016     103
2017      94
2018      58
2019      39
2020     140
2021      29
2022      17
2023      12
2024      11
2025      13
2026       3
Name: count, dtype: int64


In [7]:
df_def_2015 = dates_defaut(INTERIM / "perf_2015.csv")
n_2015, par_annee_2015 = resume_defauts(df_def_2015, n_prets=50_000)

défauts : 2,058 sur 50,000 prêts, taux = 4.12%

par déclencheur du premier mois :
declencheur
statut    2057
RA           1
Name: count, dtype: int64

par année calendaire de défaut :
date_defaut
2015      14
2016     102
2017     241
2018     145
2019     120
2020    1115
2021     112
2022      66
2023      46
2024      36
2025      42
2026      19
Name: count, dtype: int64


## 5. Pourquoi autant de défauts en 2020 ? Marquage de la forbearance (règle v2)
Le millésime 2015 compte 1 115 défauts en 2020 sur 2 058. On marque les défauts qui portent un plan d'assistance (suspension de paiement) ou un sinistre, puis on compare le taux avec et sans ces prêts. Les défauts restent comptés : seule la colonne `defaut_forbearance` est ajoutée.

In [8]:
df_def_2007 = marquer_forbearance(INTERIM / "perf_2007.csv", df_def_2007)
df_def_2015 = marquer_forbearance(INTERIM / "perf_2015.csv", df_def_2015)

for nom, df in [("2007", df_def_2007), ("2015", df_def_2015)]:
    n = len(df); sans = (~df["defaut_forbearance"]).sum()
    print(f"{nom} : taux avec = {n/50_000:.2%} | sans forbearance = {sans/50_000:.2%}")

2007 : taux avec = 16.53% | sans forbearance = 16.11%
2015 : taux avec = 4.12% | sans forbearance = 1.34%


## 6. Issue des défauts et pertes enregistrées
Combien de prêts ont une perte renseignée (`Actual Loss`) ? La perte n'est écrite que sur la dernière ligne d'un prêt, donc le nombre de lignes avec perte est un nombre de prêts.

In [9]:
perf_loss_2007 = perf_2007[perf_2007["Actual Loss"] != ""]
perf_loss_2007.shape

(4043, 35)

In [10]:
perf_loss_2015 = perf_2015[perf_2015["Actual Loss"] != ""]
perf_loss_2015.shape

(134, 35)

Répartition des défauts selon leur issue (dernière ligne du prêt) :
- `sortie 02/03/09/15 avec perte` : liquidation avec une perte enregistrée (base de la LGD) ;
- `sortie 01` : remboursement anticipé ou à l'échéance ; `sortie 16` : revente comme prêt redevenu performant ; `sortie 96` : défaut de souscription confirmé avant l'événement de crédit ;
- `encore en vie` : défaut sans sortie à la date de coupure.

In [11]:
sortie_2007 = sortie_des_defauts(INTERIM / "perf_2007.csv", df_def_2007)
sortie_2015 = sortie_des_defauts(INTERIM / "perf_2015.csv", df_def_2015)

sortie
sortie 02/03/09/15 avec perte    4043
sortie 01                        1814
sortie 16                        1088
encore en vie (pas de sortie)     445
sortie 02/03/09/15 SANS perte     443
sortie 96                         432
Name: count, dtype: int64

total : 8,265 défauts
sortie
sortie 01                        966
encore en vie (pas de sortie)    847
sortie 02/03/09/15 avec perte    134
sortie 16                        104
sortie 96                          5
sortie 02/03/09/15 SANS perte      2
Name: count, dtype: int64

total : 2,058 défauts


## 7. Défauts liquidés sans montant de perte : règlement de défaut de souscription
443 défauts du millésime 2007 sont sortis par 02/03/09/15 mais sans perte. Hypothèse : le prêteur a racheté le prêt ou indemnisé Freddie Mac (défaut de souscription ou de gestion), donc aucune perte n'est enregistrée sur le prêt. On vérifie la date de règlement sur au moins une ligne du prêt.

In [12]:
def test_defaut_souscription(perf_csv, sortie_df, id_col="Loan Identifier",
                             col="Underwriting Defect and Major Servicing Defect Settlement Date",
                             chunksize=500_000):
    """Parmi les défauts sortis en 02/03/09/15 SANS perte, combien ont une date de règlement
    de défaut de souscription renseignée sur au moins une ligne ?"""
    cible = set(sortie_df.loc[sortie_df["sortie"] == "sortie 02/03/09/15 SANS perte", id_col])
    avec_date = set()
    for ch in pd.read_csv(perf_csv, dtype=str, keep_default_na=False,
                          usecols=[id_col, col], chunksize=chunksize):
        ch = ch[ch[id_col].isin(cible)]
        avec_date |= set(ch.loc[ch[col] != "", id_col])
    print(f"{len(avec_date)} prêts sur {len(cible)} ont une date de règlement renseignée")
    return avec_date

In [13]:
ids_441 = test_defaut_souscription(INTERIM / "perf_2007.csv", sortie_2007)

441 prêts sur 443 ont une date de règlement renseignée


## 8. Échantillon retenu pour la LGD
Défauts liquidés (02/03/09/15), avec perte renseignée et sans règlement de défaut de souscription. Ces prêts restent comptés pour la PD ; ils sont seulement écartés de la LGD.

In [14]:
df_def_2007 = marquer_defaut_souscription(INTERIM / "perf_2007.csv", df_def_2007)
df_def_2015 = marquer_defaut_souscription(INTERIM / "perf_2015.csv", df_def_2015)
ids_lgd_2007 = echantillon_lgd(sortie_2007, df_def_2007)
ids_lgd_2015 = echantillon_lgd(sortie_2015, df_def_2015)

défauts : 8,265 | avec perte renseignée : 4,043 | dont retirés (règlement de défaut de souscription) : 0 | échantillon LGD : 4,043
défauts : 2,058 | avec perte renseignée : 134 | dont retirés (règlement de défaut de souscription) : 0 | échantillon LGD : 134


## 9. Part de FICO non disponible
Le code 9999 signifie « non disponible » : on mesure sa fréquence avant de le traiter comme manquant.

In [15]:
_ = part_fico_manquant(INTERIM / "orig_2007.csv")
_ = part_fico_manquant(INTERIM / "orig_2015.csv")

Classic FICO® : 38 sur 50,000 non disponibles (0.08%)
valeurs disponibles : min 300, médiane 732, max 844
Classic FICO® : 0 sur 50,000 non disponibles (0.00%)
valeurs disponibles : min 462, médiane 760, max 834


## Résumé des résultats
| | Millésime 2007 | Millésime 2015 |
|---|---|---|
| Lignes de performance | 3 015 985 | 3 501 425 |
| Défauts (règle v1) | 8 265 (16,53 %) | 2 058 (4,12 %) |
| Taux sans forbearance | 16,11 % | 1,34 % |
| Défauts avec perte enregistrée | 4 043 | 134 |
| Échantillon LGD | 4 043 | 134 |
| FICO non disponible | 38 (0,08 %) | 0 |

- Les deux fichiers sont cohérents : 50 000 prêts uniques de chaque côté, aucun orphelin.
- En 2015, la forbearance explique l'essentiel du pic de défauts de 2020 (4,12 % contre 1,34 % hors forbearance).
- Sur les 443 défauts de 2007 liquidés sans perte, 441 ont un règlement de défaut de souscription renseigné.
- La construction du panel prêt-année et les modèles de PD sont dans `credit/panel.py`, `credit/models.py` et le notebook 03.